In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
"""
catev_cnn1d_v8_training.py
==========================
Fixes three confirmed problems from v7 result analysis.

════════════════════════════════════════════════════════════════════
V7 RESULT — CONFIRMED PROBLEMS
════════════════════════════════════════════════════════════════════

✗  PROBLEM 1: Critical weight=3.5 caused class collapse
   77% of True-Emergency predicted as Critical.
   Critical recall=0.75, Emergency recall=0.18.
   Model learned "when unsure → predict Critical" to avoid 3.5x penalty.
   [FIX-1] Remove manual weight. Use sklearn balanced weights.
           Critical≈1.7, which is proportional to imbalance, not inflated.

✗  PROBLEM 2: stride=15 partially reintroduced autocorrelation
   67% window overlap → sequential windows too similar again.
   But stride=30 only gave 16k windows and model stopped at epoch 2.
   Tension: need 33% overlap AND enough training data.
   [FIX-2] stride=30 + random jitter augmentation.
           For each strided position, also sample a version with a
           random ±7 row offset. Doubles data to ~32k windows WITHOUT
           creating sequential overlap. Structural overlap stays at 33%.

✗  PROBLEM 3: Patient split still has distribution mismatch
   Val: N=21% C=21% E=57%  Test: N=31% C=10% E=57%
   57% Emergency in val/test vs 36% in train — Emergency dominates.
   [FIX-3] Improved stratification: sort patients by per-class proportion
           distance, then assign in strict interleaved order ensuring
           each split receives similar Emergency/Critical/Normal ratios.

════════════════════════════════════════════════════════════════════
FEATURE SET — 41 features (unchanged from v7)
════════════════════════════════════════════════════════════════════
  Raw vitals    (8): spo2, heart_rate, resp_rate_smoothed, sbp, dbp,
                     mbp, etco2, pulse_pressure
  s_* scaled    (8): s_spo2, s_hr, s_rr, s_sbp, s_dbp, s_mbp,
                     s_etco2, s_pp
  Slope 7m      (8): slope_7m_* per vital
  Slope 15m     (8): slope_15m_* per vital
  Combined      (9): combined_score, slope_7m_combined,
                     slope_15m_combined, roll_mean_7m, roll_std_7m,
                     roll_mean_15m, roll_std_15m,
                     roll_min_15m, roll_max_15m
"""

import sys
import warnings
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    f1_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
WINDOW          = 45     # 90s context
STRIDE          = 30     # [FIX-2] back to 30 (33% overlap) + jitter augmentation
JITTER          = 7      # [FIX-2] ±7 row random offset for augmented windows
N_TRAIN_PT      = 80
N_VAL_PT        = 10
BATCH_SIZE      = 256
MAX_EPOCHS      = 60
ES_PATIENCE     = 10
LR_INIT         = 3e-4
LR_MIN          = 1e-5
LR_PATIENCE     = 5
LR_FACTOR       = 0.4
L2_REG          = 1e-4
SEED            = 42
# [FIX-1] No manual class weights — sklearn balanced used (Critical≈1.7, not 3.5)

PURE_VITALS = [
    "spo2", "heart_rate", "resp_rate_smoothed",
    "sbp", "dbp", "mbp", "etco2", "pulse_pressure",
]

# ─────────────────────────────────────────────────────────────────────────────
# FEATURE DROP SET
# ─────────────────────────────────────────────────────────────────────────────
DROP_ALWAYS = {
    # identifiers / time
    "patient_id", "age", "time",
    # unsmoothed resp_rate
    "resp_rate",
    # z-scores (duplicate of raw vitals)
    "z_spo2","z_hr","z_rr","z_sbp","z_dbp","z_mbp","z_etco2","z_pp",
    # severity_sum
    "severity_sum",
    # text outputs
    "selected_cond_1","selected_cond_2",
    "severity_label","result_label",
    # [REQ-1] t1/t2/t3 rule flags — removed per user request
    "t1_shock_spiral","t1_resp_burnout","t1_hypercapnic",
    "t2_pulse_pressure_low","t2_widepp_highsbp","t2_resp_hemo_combo",
    "t3_hyper_emergency","t3_stable_deceiver","t3_masked_shock",
    "t3_occult_acidosis","t3_trend_decline","t3_trend_activate",
    # 2-min & 5-min slopes — too noisy
    *[f"slope_2m_{v}" for v in PURE_VITALS],
    "slope_2m_combined_score",
    *[f"slope_5m_{v}" for v in PURE_VITALS],
    "slope_5m_combined_score",
    # lag features — window captures history
    *[f"lag_15m_{v}" for v in PURE_VITALS],
    "lag_15m_combined_score",
    # short rolling windows
    "roll_mean_2m_combined","roll_std_2m_combined",
    "roll_mean_5m_combined","roll_std_5m_combined",
    # leakage
    "etco2_prev","spo2_prev","heart_rate_prev",
    # target
    "future_label",
    # NOTE: s_spo2, s_hr, s_rr, s_sbp, s_dbp, s_mbp, s_etco2, s_pp
    #       intentionally NOT in this set — kept per [REQ-2]
}


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE MATRIX
# ─────────────────────────────────────────────────────────────────────────────
def build_feature_matrix(df):
    y      = df["future_label"].astype(int)
    groups = df["patient_id"].values

    drop_cols = DROP_ALWAYS & set(df.columns)
    X = df.drop(columns=list(drop_cols))
    X = X.drop(columns=X.select_dtypes(include="object").columns.tolist(),
                errors="ignore")

    # Safety sweep — ensure t1/t2/t3 are gone
    rule_left = [c for c in X.columns if c.startswith(("t1_","t2_","t3_"))]
    if rule_left:
        print(f"  [warn] removing leftover rule cols: {rule_left}")
        X = X.drop(columns=rule_left)

    # Confirm s_* are present
    s_present = [c for c in X.columns if c.startswith("s_")]
    print(f"  s_* scaled features kept ({len(s_present)}): {s_present}")
    print(f"  Feature count: {X.shape[1]}")
    print(f"  Features: {X.columns.tolist()}")
    return X, y, groups, X.columns.tolist()


# ─────────────────────────────────────────────────────────────────────────────
# [FIX-3] STRATIFIED PATIENT SPLIT — by distribution similarity
# ─────────────────────────────────────────────────────────────────────────────
def stratified_patient_split(df, n_train, n_val, seed=SEED):
    """
    Assigns patients to splits so each split's class distribution
    matches the global distribution as closely as possible.

    Method: compute each patient's label proportion vector [p_N, p_C, p_E],
    sort patients by L1 distance from global distribution,
    then assign round-robin across train/val/test buckets.
    This ensures similar-distribution patients spread across all splits.
    """
    rng = np.random.default_rng(seed)

    # Global label proportions
    total_counts = df["future_label"].value_counts(normalize=True).sort_index()
    global_dist  = np.array([total_counts.get(i, 0) for i in range(3)])

    # Per-patient label proportions
    per_pt = (df.groupby("patient_id")["future_label"]
              .value_counts(normalize=True)
              .unstack(fill_value=0)
              .reindex(columns=[0.0, 1.0, 2.0], fill_value=0))
    per_pt.columns = [0, 1, 2]

    # L1 distance from global
    per_pt["dist"] = per_pt[[0,1,2]].apply(
        lambda row: np.abs(row.values - global_dist).sum(), axis=1)

    # Sort by distance, add small random jitter so ties are broken randomly
    per_pt["jitter"] = rng.random(len(per_pt))
    per_pt = per_pt.sort_values(["dist","jitter"])
    pids_sorted = per_pt.index.tolist()

    # Round-robin assign: most-similar patients spread across all splits
    # Buckets: 0=train, 1=val, 2=test  cycling 0,0,...,0,1,2,0,0,...
    # Weight train heavier: for every n_train+n_val+n_test patients,
    # assign n_train to train, n_val to val, n_test to test
    n_test  = len(pids_sorted) - n_train - n_val
    train_pts, val_pts, test_pts = [], [], []

    # Simple proportional round-robin
    slots = (["train"] * n_train + ["val"] * n_val + ["test"] * n_test)
    # Interleave: every group of (train+val+test) patients, distribute 1 to each
    block = n_train + n_val + n_test
    assigned = {"train": [], "val": [], "test": []}
    for i, pid in enumerate(pids_sorted):
        pos = i % block
        if pos < n_train:
            assigned["train"].append(pid)
        elif pos < n_train + n_val:
            assigned["val"].append(pid)
        else:
            assigned["test"].append(pid)

    return (set(assigned["train"]),
            set(assigned["val"]),
            set(assigned["test"]))


# ─────────────────────────────────────────────────────────────────────────────
# SLIDING WINDOWS WITH STRIDE
# ─────────────────────────────────────────────────────────────────────────────
def build_windows_for_patient(X_arr, y_arr, window=WINDOW, stride=STRIDE,
                              augment_jitter=False, rng=None):
    """
    stride=30 → 33% overlap base windows.
    augment_jitter=True → also sample one jittered version per base position
    with a random ±JITTER row offset, doubling data without sequential overlap.
    """
    n = len(X_arr)
    if n < window:
        return None, None

    base_indices = list(range(0, n - window + 1, stride))
    X_wins, y_wins = [], []

    for i in base_indices:
        X_wins.append(X_arr[i:i+window])
        y_wins.append(y_arr[i+window-1])

        if augment_jitter and rng is not None:
            # Random offset: stay within valid bounds
            offset = rng.integers(-JITTER, JITTER + 1)
            j = i + offset
            j = max(0, min(j, n - window))
            # Only add if this position isn't already a base index
            if j not in base_indices:
                X_wins.append(X_arr[j:j+window])
                y_wins.append(y_arr[j+window-1])

    X_win = np.stack(X_wins).astype(np.float32)
    y_win = np.array(y_wins, dtype=np.int32)
    return X_win, y_win


def build_windows(X_df, y_ser, groups, window=WINDOW, stride=STRIDE,
                  desc="set", shuffle=False, augment=False):
    """
    augment=True (train only): adds jittered copies to double data
    while keeping structural overlap at 33%.
    """
    all_X, all_y = [], []
    rng = np.random.default_rng(SEED) if augment else None

    for pid in np.unique(groups):
        mask = groups == pid
        Xw, yw = build_windows_for_patient(
            X_df.values[mask], y_ser.values[mask],
            window, stride, augment_jitter=augment, rng=rng)
        if Xw is None:
            continue
        all_X.append(Xw)
        all_y.append(yw)

    X_out = np.concatenate(all_X, axis=0)
    y_out = np.concatenate(all_y, axis=0)

    if shuffle:
        idx = np.random.default_rng(SEED).permutation(len(X_out))
        X_out, y_out = X_out[idx], y_out[idx]

    aug_note = " +jitter_aug" if augment else ""
    print(f"  {desc}: {X_out.shape[0]:,} windows  "
          f"shape={X_out.shape}  stride={stride}{aug_note}"
          + ("  [shuffled ✓]" if shuffle else ""))
    return X_out, y_out


# ─────────────────────────────────────────────────────────────────────────────
# CNN  — LayerNorm, multi-scale branches
# ─────────────────────────────────────────────────────────────────────────────
def build_cnn(n_features, n_classes, window=WINDOW):
    import tensorflow as tf
    from tensorflow.keras import layers, Model, regularizers

    reg = regularizers.l2(L2_REG)
    inp = tf.keras.Input(shape=(window, n_features), name="vitals_window")

    # Branch A: fine detail (kernel=3, 6s)
    a = layers.Conv1D(64, 3,  padding="causal", activation="relu",
                      kernel_regularizer=reg)(inp)
    a = layers.LayerNormalization()(a)
    a = layers.Dropout(0.25)(a)

    # Branch B: medium pattern (kernel=7, 14s)
    b = layers.Conv1D(64, 7,  padding="causal", activation="relu",
                      kernel_regularizer=reg)(inp)
    b = layers.LayerNormalization()(b)
    b = layers.Dropout(0.25)(b)

    # Branch C: slow trend (kernel=15, 30s)
    c = layers.Conv1D(64, 15, padding="causal", activation="relu",
                      kernel_regularizer=reg)(inp)
    c = layers.LayerNormalization()(c)
    c = layers.Dropout(0.25)(c)

    # Merge → (batch, window, 192)
    x = layers.Concatenate(axis=-1)([a, b, c])

    # Refinement 1
    x = layers.Conv1D(128, 5, padding="causal", activation="relu",
                      kernel_regularizer=reg)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(0.25)(x)

    # Refinement 2
    x = layers.Conv1D(128, 3, padding="causal", activation="relu",
                      kernel_regularizer=reg)(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu", kernel_regularizer=reg)(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(n_classes, activation="softmax", name="severity")(x)

    return Model(inputs=inp, outputs=out, name="CatevCNN1D_v8")


# ─────────────────────────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_split(model, X_win, y_true, label="Split", prefix="catev_cnn"):
    y_pred = np.argmax(
        model.predict(X_win, batch_size=512, verbose=0), axis=1)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    ba = balanced_accuracy_score(y_true, y_pred)
    print(f"\n  {label}  macro-F1={f1:.4f}  bal-acc={ba:.4f}")

    report = classification_report(
        y_true, y_pred,
        target_names=["Normal(0)","Critical(1)","Emergency(2)"],
        zero_division=0)
    print(report)

    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix (counts):")
    print(pd.DataFrame(cm,
        index=["True-Normal","True-Critical","True-Emergency"],
        columns=["Pred-Normal","Pred-Critical","Pred-Emergency"]))

    cm_norm = (cm.astype(float) / cm.sum(axis=1, keepdims=True)).round(4)
    print("\nNormalised (row %):")
    print(pd.DataFrame(cm_norm,
        index=["True-Normal","True-Critical","True-Emergency"],
        columns=["Pred-Normal","Pred-Critical","Pred-Emergency"]))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, mat, title, fmt in zip(
        axes, [cm, cm_norm],
        [f"{label} — Counts", f"{label} — Row %"],
        [".0f", ".2f"]
    ):
        ConfusionMatrixDisplay(
            confusion_matrix=mat,
            display_labels=["Normal","Critical","Emergency"]
        ).plot(ax=ax, colorbar=True,
               cmap="Blues" if mat is cm else "YlOrRd",
               values_format=fmt)
        ax.set_title(title, fontsize=11)
    plt.tight_layout()
    path = f"{prefix}_{label.lower().replace(' ','_')}_cm.png"
    fig.savefig(path, dpi=150); plt.close(fig)
    print(f"  → {path}")
    return f1, ba, cm, report, y_pred


def plot_history(history, prefix):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history.history["loss"],     label="train loss")
    axes[0].plot(history.history["val_loss"], label="val loss")
    axes[0].axhline(1.099, color="gray", ls="--", alpha=0.5,
                    label="random (1.099)")
    axes[0].set_title("Loss per epoch")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    acc = "sparse_categorical_accuracy"
    axes[1].plot(history.history[acc],           label="train acc")
    axes[1].plot(history.history[f"val_{acc}"],  label="val acc")
    axes[1].set_title("Accuracy per epoch")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = f"{prefix}_history.png"
    fig.savefig(path, dpi=150); plt.close(fig)
    print(f"  → {path}")


# ─────────────────────────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def run(input_csv, output_prefix="catev_cnn_v7"):
    import tensorflow as tf
    from tensorflow.keras import optimizers, callbacks as K_callbacks

    tf.random.set_seed(SEED); np.random.seed(SEED)

    print(f"\n{'='*65}")
    print("  CatevCode CNN1D v7 — Analysis-Driven Fixes")
    print(f"{'='*65}\n")

    df = pd.read_csv(input_csv)
    print(f"Loaded  : {df.shape[0]:,} rows × {df.shape[1]} cols")
    n_patients  = df["patient_id"].nunique()
    rows_per_pt = df.groupby("patient_id").size()
    print(f"Patients: {n_patients}  "
          f"min={rows_per_pt.min():,}  "
          f"median={rows_per_pt.median():,.0f}  "
          f"max={rows_per_pt.max():,}")

    print("\nTarget distribution (future_label):")
    vc = df["future_label"].value_counts().sort_index()
    for lbl, cnt in vc.items():
        name = {0:"Normal",1:"Critical",2:"Emergency"}.get(lbl, str(lbl))
        print(f"  {lbl} ({name:>9}): {cnt:7,d}  ({100*cnt/len(df):5.1f}%)")

    # Features
    X, y, groups, feat_names = build_feature_matrix(df)
    n_features = X.shape[1]
    print(f"\nFinal feature count : {n_features}")

    # [FIX-3] Distribution-similarity patient split
    print(f"\n[Split] Distribution-similarity stratified assignment")
    train_pids, val_pids, test_pids = stratified_patient_split(
        df, N_TRAIN_PT, N_VAL_PT, SEED)
    print(f"  Train={len(train_pids)} | Val={len(val_pids)} "
          f"| Test={len(test_pids)} patients")

    tr_mask = np.isin(groups, list(train_pids))
    va_mask = np.isin(groups, list(val_pids))
    te_mask = np.isin(groups, list(test_pids))

    X_tr, y_tr, g_tr = X[tr_mask],  y[tr_mask],  groups[tr_mask]
    X_va, y_va, g_va = X[va_mask],  y[va_mask],  groups[va_mask]
    X_te, y_te, g_te = X[te_mask],  y[te_mask],  groups[te_mask]

    print("  Class distributions after stratification:")
    for nm, ys in [("Train", y_tr), ("Val  ", y_va), ("Test ", y_te)]:
        cls, cts = np.unique(ys, return_counts=True)
        dist = "  ".join(f"{['N','C','E'][int(c)]}={100*n/len(ys):.1f}%"
                         for c, n in zip(cls, cts))
        print(f"    {nm}: [{dist}]  ← should be close across all three")

    # StandardScaler
    print("\n[Scale] StandardScaler fit on train rows ...")
    scaler = StandardScaler()
    X_tr_s = pd.DataFrame(scaler.fit_transform(X_tr),
                           columns=feat_names, index=X_tr.index)
    X_va_s = pd.DataFrame(scaler.transform(X_va),
                           columns=feat_names, index=X_va.index)
    X_te_s = pd.DataFrame(scaler.transform(X_te),
                           columns=feat_names, index=X_te.index)

    # Windows
    print(f"\n[Windows]  WINDOW={WINDOW} rows ({WINDOW*2}s)  "
          f"STRIDE={STRIDE} rows ({STRIDE*2}s)  "
          f"overlap={(WINDOW-STRIDE)/WINDOW*100:.0f}%  jitter=±{JITTER}")
    X_tr_w, y_tr_w = build_windows(
        X_tr_s, y_tr, g_tr, WINDOW, STRIDE, "Train",
        shuffle=True, augment=True)       # [FIX-2] jitter augmentation on train
    X_va_w, y_va_w = build_windows(
        X_va_s, y_va, g_va, WINDOW, STRIDE, "Val  ",
        shuffle=False, augment=False)
    X_te_w, y_te_w = build_windows(
        X_te_s, y_te, g_te, WINDOW, STRIDE, "Test ",
        shuffle=False, augment=False)

    # [FIX-1] Balanced class weights — no manual inflation
    classes_ = np.unique(y_tr_w)
    cw_arr   = compute_class_weight("balanced", classes=classes_, y=y_tr_w)
    cw_dict  = {int(c): float(w) for c, w in zip(classes_, cw_arr)}
    print(f"\nClass weights (balanced — no manual inflation):")
    for c, w in cw_dict.items():
        name = {0:"Normal",1:"Critical",2:"Emergency"}[c]
        print(f"  class {c} ({name:>9}): {w:.4f}")

    # Model
    model = build_cnn(n_features, 3, WINDOW)
    model.summary()
    model.compile(
        optimizer=optimizers.Adam(LR_INIT),
        loss="sparse_categorical_crossentropy",
        metrics=["sparse_categorical_accuracy"],
    )

    cb_list = [
        K_callbacks.EarlyStopping(
            monitor="val_loss", patience=ES_PATIENCE,
            restore_best_weights=True, verbose=1),
        K_callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=LR_FACTOR,
            patience=LR_PATIENCE, min_lr=LR_MIN, verbose=1),
        K_callbacks.ModelCheckpoint(
            f"{output_prefix}_best.keras",
            monitor="val_loss", save_best_only=True, verbose=0),
    ]

    print(f"\n[Train] batch={BATCH_SIZE}  max_epochs={MAX_EPOCHS}"
          f"  LR={LR_INIT}  window={WINDOW}  stride={STRIDE}"
          f"  features={n_features}")
    print(f"  Fixes: stride=30 ✓  jitter_aug ✓  balanced_weights ✓"
          f"  distribution_split ✓")

    history = model.fit(
        X_tr_w, y_tr_w,
        validation_data=(X_va_w, y_va_w),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw_dict,
        callbacks=cb_list,
        verbose=1,
    )
    print(f"  Stopped at epoch {len(history.history['loss'])}")
    plot_history(history, output_prefix)

    # Evaluate
    print("\n" + "="*55 + "\n  VALIDATION RESULTS\n" + "="*55)
    val_f1, val_ba, _, _, _ = evaluate_split(
        model, X_va_w, y_va_w, "Validation", output_prefix)

    print("\n" + "="*55 + "\n  TEST RESULTS\n" + "="*55)
    test_f1, test_ba, cm_test, report_test, _ = evaluate_split(
        model, X_te_w, y_te_w, "Test", output_prefix)

    # Save
    model.save(f"{output_prefix}_model.keras")
    with open(f"{output_prefix}_meta.pkl", "wb") as fh:
        pickle.dump({
            "scaler":       scaler,
            "features":     feat_names,
            "window":       WINDOW,
            "stride":       STRIDE,
            "n_classes":    3,
            "class_weights": cw_dict,
            "class_map":    {0:"Normal",1:"Critical",2:"Emergency"},
        }, fh)
    print(f"\nModel  → {output_prefix}_model.keras")
    print(f"Meta   → {output_prefix}_meta.pkl")

    # Report
    rpath = f"{output_prefix}_report.txt"
    cm_df = pd.DataFrame(cm_test,
        index=["True-Normal","True-Critical","True-Emergency"],
        columns=["Pred-Normal","Pred-Critical","Pred-Emergency"])
    cm_norm_df = pd.DataFrame(
        (cm_test.astype(float)/cm_test.sum(axis=1,keepdims=True)).round(4),
        index=["True-Normal","True-Critical","True-Emergency"],
        columns=["Pred-Normal","Pred-Critical","Pred-Emergency"])

    with open(rpath, "w") as fh:
        fh.write("CATEV CNN1D v8 — V7 ANALYSIS FIXES\n")
        fh.write("="*65 + "\n\n")
        fh.write(f"Dataset  : {input_csv}\n")
        fh.write(f"Rows     : {len(df):,}   Patients : {n_patients}\n")
        fh.write(f"Features : {n_features}\n")
        fh.write(f"Window   : {WINDOW} rows ({WINDOW*2}s)\n")
        fh.write(f"Stride   : {STRIDE} rows ({STRIDE*2}s)  "
                 f"overlap={(WINDOW-STRIDE)/WINDOW*100:.0f}%\n")
        fh.write(f"Jitter   : ±{JITTER} rows augmentation on train\n\n")

        fh.write("V7 PROBLEMS → V8 FIXES\n")
        fh.write("-"*50 + "\n")
        fh.write("[FIX-1] Critical weight 3.5→balanced(~1.7)\n")
        fh.write("        77% True-Emergency predicted Critical — class collapsed\n")
        fh.write(f"[FIX-2] stride=15→30 + jitter±{JITTER} augmentation\n")
        fh.write("        stride=15 reintroduced 67% overlap autocorrelation\n")
        fh.write("        jitter doubles train windows without sequential overlap\n")
        fh.write("[FIX-3] Improved distribution-similarity split\n")
        fh.write("        Val had 57% Emergency vs 36% train\n\n")

        fh.write(f"\nVAL   macro-F1={val_f1:.4f}  bal-acc={val_ba:.4f}\n")
        fh.write(f"TEST  macro-F1={test_f1:.4f}  bal-acc={test_ba:.4f}\n\n")
        fh.write("TEST — CLASSIFICATION REPORT\n" + report_test + "\n")
        fh.write("CONFUSION MATRIX (counts)\n" + cm_df.to_string() + "\n\n")
        fh.write("CONFUSION MATRIX (row %)\n" + cm_norm_df.to_string() + "\n\n")
        fh.write("FEATURES USED\n" +
                 "\n".join(f"  {fn}" for fn in feat_names) + "\n")

    print(f"Report → {rpath}\nDone.\n")
    return model


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
def _is_notebook():
    try:
        return get_ipython().__class__.__name__ in (   # type: ignore
            "ZMQInteractiveShell","TerminalInteractiveShell","Shell")
    except NameError:
        return False

if __name__ == "__main__":
    if _is_notebook():
        input_csv = "/kaggle/input/datasets/arjunmahesh09999/master-data/Master_DATASET.csv"
        prefix    = "catev_cnn_v8"
    else:
        args      = [a for a in sys.argv[1:] if not a.startswith("-")]
        input_csv = args[0] if args else "/kaggle/input/datasets/arjunmahesh09999/master-data/Master_DATASET.csv"
        prefix    = args[1] if len(args) >= 2 else "catev_cnn_v8"
    run(input_csv, prefix)


  CatevCode CNN1D v7 — Analysis-Driven Fixes

Loaded  : 620,257 rows × 99 cols
Patients: 103  min=1,110  median=5,978  max=15,240

Target distribution (future_label):
  0.0 (   Normal): 244,449  ( 39.4%)
  1.0 ( Critical): 118,468  ( 19.1%)
  2.0 (Emergency): 257,340  ( 41.5%)
  s_* scaled features kept (8): ['s_spo2', 's_hr', 's_rr', 's_sbp', 's_dbp', 's_mbp', 's_etco2', 's_pp']
  Feature count: 41
  Features: ['dbp', 'mbp', 'heart_rate', 'sbp', 'spo2', 'etco2', 'pulse_pressure', 'slope_7m_spo2', 'slope_15m_spo2', 'slope_7m_heart_rate', 'slope_15m_heart_rate', 'resp_rate_smoothed', 'slope_7m_resp_rate_smoothed', 'slope_15m_resp_rate_smoothed', 'slope_7m_sbp', 'slope_15m_sbp', 'slope_7m_dbp', 'slope_15m_dbp', 'slope_7m_mbp', 'slope_15m_mbp', 'slope_7m_etco2', 'slope_15m_etco2', 'slope_7m_pulse_pressure', 'slope_15m_pulse_pressure', 's_spo2', 's_hr', 's_rr', 's_sbp', 's_dbp', 's_mbp', 's_etco2', 's_pp', 'combined_score', 'slope_7m_combined_score', 'slope_15m_combined_score', 'roll_mean

Model: "CatevCNN1D_v8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ vitals_window       │ (None, 45, 41)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_10 (Conv1D)  │ (None, 45, 64)    │      7,936 │ vitals_window[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_11 (Conv1D)  │ (None, 45, 64)    │     18,432 │ vitals_window[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 45, 64)    │     39,424 │ vitals_window[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 45, 64)    │        128 │ conv1d_10[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 45, 64)    │        128 │ conv1d_11[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 45, 64)    │        128 │ conv1d_12[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 45, 64)    │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 45, 64)    │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 45, 64)    │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 45, 192)   │          0 │ dropout_12[0][0], │
│ (Concatenate)       │                   │            │ dropout_13[0][0], │
│                     │                   │            │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 45, 128)   │    123,008 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 45, 128)   │        256 │ conv1d_13[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 45, 128)   │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 45, 128)   │     49,280 │ dropout_15[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 45, 128)   │        256 │ conv1d_14[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 45, 128)   │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout_16[0][0]  │
│ (GlobalAveragePool… │                   │            │                 

 Total params: 247,427 (966.51 KB)

 Trainable params: 247,427 (966.51 KB)

 Non-trainable params: 0 (0.00 B)


[Train] batch=256  max_epochs=60  LR=0.0003  window=45  stride=30  features=41
  Fixes: stride=30 ✓  jitter_aug ✓  balanced_weights ✓  distribution_split ✓
Epoch 1/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 36s 259ms/step - loss: 1.1474 - sparse_categorical_accuracy: 0.4449 - val_loss: 1.0009 - val_sparse_categorical_accuracy: 0.5404 - learning_rate: 3.0000e-04
Epoch 2/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 31s 257ms/step - loss: 1.0195 - sparse_categorical_accuracy: 0.5392 - val_loss: 1.0545 - val_sparse_categorical_accuracy: 0.5027 - learning_rate: 3.0000e-04
Epoch 3/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 30s 254ms/step - loss: 0.9658 - sparse_categorical_accuracy: 0.5830 - val_loss: 1.1117 - val_sparse_categorical_accuracy: 0.4663 - learning_rate: 3.0000e-04
Epoch 4/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 31s 260ms/step - loss: 0.9124 - sparse_categorical_accuracy: 0.6187 - val_loss: 1.2188 - val_sparse_categorical_accuracy: 0.4264 - learning_rate: 3.0000e-04
Epoch 5/60
119/119 ━━━━━━━━━━━━━━━━━━━━ 30s 252ms/step 